## Try to make self healing wifi controller code, auto connect to hotspot on disconnect etc. 

# Final iteration.  Make things defiend at the top, set trim to 0 by default, 

## Get rid of ping check

In [ ]:
import pygame
import win32gui
import win32con
import socket
import time
import subprocess
import os

# ==========================
# USER SETTINGS
# ==========================

# Default trim offset (-1.0 to +1.0)
DEFAULT_TRIM = -0.30        # slider starts at 0, but this is your saved baseline
MAX_TRIM = 155             # max PWM to subtract when trim = ±1.0

# ESP32 AP WiFi
ROVER_SSID = "Rover"
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

# ==========================
# UDP SOCKET
# ==========================
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setblocking(False)

def send_udp(speedA, dirA, speedB, dirB):
    msg = f"{speedA},{dirA},{speedB},{dirB}"
    try:
        sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))
    except:
        pass

def stop_all():
    send_udp(0, 0, 0, 0)

# ==========================
# WIFI CONTROL (WINDOWS)
# ==========================
def wifi_connect(ssid):
    try:
        subprocess.run(["netsh", "wlan", "connect", f"name={ssid}"], check=False)
    except:
        pass

def wifi_disconnect():
    try:
        subprocess.run(["netsh", "wlan", "disconnect"], check=False)
    except:
        pass

# ==========================
# Pygame Setup
# ==========================
pygame.init()
pygame.font.init()
clock = pygame.time.Clock()

background = pygame.image.load("nes.png")
image_width, image_height = background.get_size()

screen = pygame.display.set_mode((image_width, image_height), pygame.RESIZABLE)
pygame.display.set_caption("NES Controller Window")

hwnd = pygame.display.get_wm_info()["window"]
pygame.display.flip()
pygame.time.wait(100)

left, top, right, bottom = win32gui.GetWindowRect(hwnd)
window_width = right - left
window_height = bottom - top

display_info = pygame.display.Info()
screen_width = display_info.current_w

x_pos = screen_width - window_width
y_pos = 0

win32gui.SetWindowPos(
    hwnd,
    win32con.HWND_TOPMOST,
    x_pos,
    y_pos,
    window_width,
    window_height,
    win32con.SWP_SHOWWINDOW
)

# ==========================
# Text Printer
# ==========================
class TextPrint:
    def __init__(self):
        self.reset()
        self.font = pygame.font.Font(None, 25)

    def tprint(self, screen, text):
        text_bitmap = self.font.render(text, True, (0, 0, 0))
        screen.blit(text_bitmap, (self.x, self.y))
        self.y += self.line_height

    def reset(self):
        self.x = 10
        self.y = 10
        self.line_height = 15

    def indent(self):
        self.x += 10

    def unindent(self):
        self.x -= 10

text_print = TextPrint()

# ==========================
# Joystick Setup
# ==========================
pygame.joystick.init()
joysticks = {}

for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# ==========================
# Trim Slider Setup
# ==========================
ALIGN_OFFSET = DEFAULT_TRIM   # slider now starts at the default trim value

SLIDER_X = 50
SLIDER_Y = image_height - 40
SLIDER_W = image_width - 200
SLIDER_H = 10
SLIDER_HANDLE_W = 20

dragging_slider = False

# Reset Trim Button
RESET_BTN_X = SLIDER_X + SLIDER_W + 30
RESET_BTN_Y = SLIDER_Y - 10
RESET_BTN_W = 80
RESET_BTN_H = 30


# ==========================
# Main Loop
# ==========================
done = False

while not done:

    # ==========================
    # EVENT HANDLING
    # ==========================
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True

        # Slider mouse events
        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = pygame.mouse.get_pos()

            # Slider drag
            if SLIDER_Y - 10 <= my <= SLIDER_Y + 20:
                dragging_slider = True

            # Reset trim button
            if RESET_BTN_X <= mx <= RESET_BTN_X + RESET_BTN_W and RESET_BTN_Y <= my <= RESET_BTN_Y + RESET_BTN_H:
                ALIGN_OFFSET = DEFAULT_TRIM

        if event.type == pygame.MOUSEBUTTONUP:
            dragging_slider = False

        elif event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy

        elif event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                del joysticks[event.instance_id]

    # Update slider
    if dragging_slider:
        mx, my = pygame.mouse.get_pos()
        pos = max(SLIDER_X, min(mx, SLIDER_X + SLIDER_W))
        ALIGN_OFFSET = ((pos - SLIDER_X) / SLIDER_W) * 2 - 1
        ALIGN_OFFSET = round(ALIGN_OFFSET, 2)

    # Clear screen
    screen.fill((255, 255, 255))
    screen.blit(background, (0, 0))
    text_print.reset()

    # Default motor values
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    # ==========================
    # NES INPUTS (RED DOT LOGIC)
    # ==========================
    for joystick in joysticks.values():

        axis_0_val = joystick.get_axis(0)
        axis_4_val = joystick.get_axis(4)
        threshold = 0.9

        UP    = (axis_4_val <= -threshold)
        DOWN  = (axis_4_val >= threshold)
        LEFT  = (axis_0_val <= -threshold)
        RIGHT = (axis_0_val >= threshold)

        # Draw red dots
        if LEFT:  pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)
        if RIGHT: pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)
        if UP:    pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)
        if DOWN:  pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)

        # ==========================
        # TURNING + TRIM LOGIC
        # ==========================

        # Base movement
        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1

        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1

        # Turning in place
        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1

        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1

        # Nothing pressed
        else:
            L_speed, L_dir = 0, 0
            R_speed, R_dir = 0, 0

        # Turning while moving
        if UP or DOWN:
            if RIGHT:
                R_speed = int(R_speed * 0.5)
            if LEFT:
                L_speed = int(L_speed * 0.5)

        # ==========================
        # APPLY TRIM IN PWM UNITS
        # ==========================
        trim_pwm = int(MAX_TRIM * abs(ALIGN_OFFSET))

        if ALIGN_OFFSET > 0:
            R_speed = max(0, R_speed - trim_pwm)
        elif ALIGN_OFFSET < 0:
            L_speed = max(0, L_speed - trim_pwm)

        # Clamp
        R_speed = max(0, min(255, R_speed))
        L_speed = max(0, min(255, L_speed))

    # ==========================
    # SEND UDP PACKET
    # ==========================
    send_udp(L_speed, L_dir, R_speed, R_dir)

    # ==========================
    # Draw Trim Slider
    # ==========================
    pygame.draw.rect(screen, (50, 50, 50), (SLIDER_X, SLIDER_Y, SLIDER_W, SLIDER_H))

    handle_x = SLIDER_X + int((ALIGN_OFFSET + 1) / 2 * SLIDER_W)
    pygame.draw.rect(screen, (200, 0, 0),
                     (handle_x - SLIDER_HANDLE_W//2, SLIDER_Y - 5,
                      SLIDER_HANDLE_W, SLIDER_H + 10))

    text_print.tprint(screen, f"Trim: {ALIGN_OFFSET:+.2f}")

    # ==========================
    # Draw Reset Trim Button
    # ==========================
    pygame.draw.rect(screen, (180, 180, 180),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H))
    pygame.draw.rect(screen, (0, 0, 0),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H), 2)

    reset_text = pygame.font.Font(None, 24).render("RESET", True, (0, 0, 0))
    screen.blit(reset_text, (RESET_BTN_X + 10, RESET_BTN_Y + 5))

    pygame.display.flip()
    clock.tick(30)

stop_all()
pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad


In [ ]:
### Invert motor sides to fix left being wired to right
import pygame
import win32gui
import win32con
import socket
import time
import subprocess
import os

# ==========================
# USER SETTINGS
# ==========================

DEFAULT_TRIM = -0.30
MAX_TRIM = 155

ROVER_SSID = "Rover"
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

# ==========================
# UDP SOCKET
# ==========================
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setblocking(False)

def send_udp(speedA, dirA, speedB, dirB):
    msg = f"{speedA},{dirA},{speedB},{dirB}"
    try:
        sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))
    except:
        pass

def stop_all():
    send_udp(0, 0, 0, 0)

# ==========================
# WIFI CONTROL (WINDOWS)
# ==========================
def wifi_connect(ssid):
    try:
        subprocess.run(["netsh", "wlan", "connect", f"name={ssid}"], check=False)
    except:
        pass

def wifi_disconnect():
    try:
        subprocess.run(["netsh", "wlan", "disconnect"], check=False)
    except:
        pass

# ==========================
# Pygame Setup
# ==========================
pygame.init()
pygame.font.init()
clock = pygame.time.Clock()

background = pygame.image.load("nes.png")
image_width, image_height = background.get_size()

screen = pygame.display.set_mode((image_width, image_height), pygame.RESIZABLE)
pygame.display.set_caption("NES Controller Window")

hwnd = pygame.display.get_wm_info()["window"]
pygame.display.flip()
pygame.time.wait(100)

left, top, right, bottom = win32gui.GetWindowRect(hwnd)
window_width = right - left
window_height = bottom - top

display_info = pygame.display.Info()
screen_width = display_info.current_w

x_pos = screen_width - window_width
y_pos = 0

win32gui.SetWindowPos(
    hwnd,
    win32con.HWND_TOPMOST,
    x_pos,
    y_pos,
    window_width,
    window_height,
    win32con.SWP_SHOWWINDOW
)

# ==========================
# Text Printer
# ==========================
class TextPrint:
    def __init__(self):
        self.reset()
        self.font = pygame.font.Font(None, 25)

    def tprint(self, screen, text):
        text_bitmap = self.font.render(text, True, (0, 0, 0))
        screen.blit(text_bitmap, (self.x, self.y))
        self.y += self.line_height

    def reset(self):
        self.x = 10
        self.y = 10
        self.line_height = 15

text_print = TextPrint()

# ==========================
# Joystick Setup
# ==========================
pygame.joystick.init()
joysticks = {}

for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# ==========================
# Trim Slider Setup
# ==========================
ALIGN_OFFSET = DEFAULT_TRIM

SLIDER_X = 50
SLIDER_Y = image_height - 40
SLIDER_W = image_width - 200
SLIDER_H = 10
SLIDER_HANDLE_W = 20

dragging_slider = False

RESET_BTN_X = SLIDER_X + SLIDER_W + 30
RESET_BTN_Y = SLIDER_Y - 10
RESET_BTN_W = 80
RESET_BTN_H = 30

# ==========================
# Main Loop
# ==========================
done = False

while not done:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True

        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = pygame.mouse.get_pos()

            if SLIDER_Y - 10 <= my <= SLIDER_Y + 20:
                dragging_slider = True

            if RESET_BTN_X <= mx <= RESET_BTN_X + RESET_BTN_W and RESET_BTN_Y <= my <= RESET_BTN_Y + RESET_BTN_H:
                ALIGN_OFFSET = DEFAULT_TRIM

        if event.type == pygame.MOUSEBUTTONUP:
            dragging_slider = False

        elif event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy

        elif event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                del joysticks[event.instance_id]

    if dragging_slider:
        mx, my = pygame.mouse.get_pos()
        pos = max(SLIDER_X, min(mx, SLIDER_X + SLIDER_W))
        ALIGN_OFFSET = ((pos - SLIDER_X) / SLIDER_W) * 2 - 1
        ALIGN_OFFSET = round(ALIGN_OFFSET, 2)

    screen.fill((255, 255, 255))
    screen.blit(background, (0, 0))
    text_print.reset()

    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    for joystick in joysticks.values():

        axis_0_val = joystick.get_axis(0)
        axis_4_val = joystick.get_axis(4)
        threshold = 0.9

        UP    = (axis_4_val <= -threshold)
        DOWN  = (axis_4_val >= threshold)
        LEFT  = (axis_0_val <= -threshold)
        RIGHT = (axis_0_val >= threshold)

        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1

        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1

        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1

        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1

        else:
            L_speed, L_dir = 0, 0
            R_speed, R_dir = 0, 0

        if UP or DOWN:
            if RIGHT:
                R_speed = int(R_speed * 0.5)
            if LEFT:
                L_speed = int(L_speed * 0.5)

        trim_pwm = int(MAX_TRIM * abs(ALIGN_OFFSET))

        if ALIGN_OFFSET > 0:
            R_speed = max(0, R_speed - trim_pwm)
        elif ALIGN_OFFSET < 0:
            L_speed = max(0, L_speed - trim_pwm)

        R_speed = max(0, min(255, R_speed))
        L_speed = max(0, min(255, L_speed))

    # ==========================
    # *** REVERSED MOTORS HERE ***
    # ==========================
    send_udp(R_speed, R_dir, L_speed, L_dir)

    pygame.draw.rect(screen, (50, 50, 50), (SLIDER_X, SLIDER_Y, SLIDER_W, SLIDER_H))

    handle_x = SLIDER_X + int((ALIGN_OFFSET + 1) / 2 * SLIDER_W)
    pygame.draw.rect(screen, (200, 0, 0),
                     (handle_x - SLIDER_HANDLE_W//2, SLIDER_Y - 5,
                      SLIDER_HANDLE_W, SLIDER_H + 10))

    text_print.tprint(screen, f"Trim: {ALIGN_OFFSET:+.2f}")

    pygame.draw.rect(screen, (180, 180, 180),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H))
    pygame.draw.rect(screen, (0, 0, 0),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H), 2)

    reset_text = pygame.font.Font(None, 24).render("RESET", True, (0, 0, 0))
    screen.blit(reset_text, (RESET_BTN_X + 10, RESET_BTN_Y + 5))

    pygame.display.flip()
    clock.tick(30)

stop_all()
pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad
